In [ ]:
import os, glob, zipfile, shutil, random, yaml
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from skimage.feature import local_binary_pattern, graycomatrix, graycoprops

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
random.seed(42); np.random.seed(42)

# OpenCV lee en BGR; matplotlib muestra en RGB. Helper para no confundirse:
def show(img, title="", cmap=None, ax=None):
    ax = ax or plt.gca()
    if img.ndim == 3:
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    else:
        ax.imshow(img, cmap=cmap or "gray")
    ax.set_title(title); ax.axis("off")

print("Entorno listo. OpenCV:", cv2.__version__)

In [ ]:
# Ruta de  ZIP
ZIP_PATH = r"C:\Users\PAULA\Downloads\Mealybug.v8-mealybug-v3.yolov11.zip"
DATASET_DIR = Path("dataset")   # aquí se extraerá

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(DATASET_DIR)

print("Extraído en:", DATASET_DIR.resolve())
# Ver estructura de carpetas
for root, dirs, files in os.walk(DATASET_DIR):
    nivel = root.replace(str(DATASET_DIR), "").count(os.sep)
    print("  " * nivel + f"[{Path(root).name}] ({len(files)} archivos)")
    if nivel >= 1 and len(files) > 0:
        print("  " * (nivel+1) + f"ej: {files[0]}")

In [ ]:
# Buscar data.yaml (a veces queda dentro de una subcarpeta)
yaml_path = next(DATASET_DIR.rglob("data.yaml"))
with open(yaml_path) as f:
    data_cfg = yaml.safe_load(f)

ROOT = yaml_path.parent   # carpeta que contiene train/valid/test
print("data.yaml:", yaml_path)
print("Clases:", data_cfg["names"])
print("Nº clases:", data_cfg.get("nc", len(data_cfg["names"])))

CLASSES = data_cfg["names"]                 # ej: ['Grape','Leaf','Mealybug','Rot']
CLASS_ID = {name: i for i, name in enumerate(CLASSES)}

# Reescribir data.yaml con rutas ABSOLUTAS (evita errores al entrenar)
data_cfg["train"] = str((ROOT / "train" / "images").resolve())
data_cfg["val"]   = str((ROOT / "valid" / "images").resolve())
data_cfg["test"]  = str((ROOT / "test"  / "images").resolve()) if (ROOT/"test").exists() else data_cfg["val"]
DATA_YAML = ROOT / "data_fixed.yaml"
with open(DATA_YAML, "w") as f:
    yaml.safe_dump(data_cfg, f)
print("Guardado:", DATA_YAML)

In [ ]:
def listar(split):
    """Devuelve listas de rutas de imágenes y labels de un split."""
    img_dir = ROOT / split / "images"
    lbl_dir = ROOT / split / "labels"
    imgs = sorted(glob.glob(str(img_dir / "*.*")))
    lbls = [str(lbl_dir / (Path(p).stem + ".txt")) for p in imgs]
    return imgs, lbls

def leer_labels(lbl_path):
    """Lee un .txt YOLO -> lista de (clase, xc, yc, w, h) normalizados."""
    if not os.path.exists(lbl_path):
        return []
    out = []
    for line in open(lbl_path):
        p = line.split()
        if len(p) == 5:
            c, xc, yc, w, h = int(p[0]), *map(float, p[1:])
            out.append((c, xc, yc, w, h))
    return out

train_imgs, train_lbls = listar("train")
val_imgs, val_lbls = listar("valid")
print(f"Train: {len(train_imgs)} imágenes | Val: {len(val_imgs)} imágenes")

# Cargar una imagen de ejemplo que usaremos en todo el pipeline
SAMPLE = cv2.imread(train_imgs[0])
print("Imagen ejemplo:", train_imgs[0], "->", SAMPLE.shape)
plt.figure(); show(SAMPLE, "Imagen de ejemplo"); plt.show()

In [ ]:
import albumentations as A
print("albumentations OK:", A.__version__)

In [ ]:
# ===== NUEVA CELDA: función de preprocesamiento (colocar ANTES de la celda 5) =====
def preprocesar(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    L, A, B = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    L = clahe.apply(L)
    img = cv2.cvtColor(cv2.merge([L, A, B]), cv2.COLOR_LAB2BGR)
    img = cv2.bilateralFilter(img, 9, 75, 75)
    return img

In [ ]:
import cv2
import albumentations as A

In [ ]:
transform = A.Compose([

    A.HorizontalFlip(p=0.5),

    A.Rotate(
        limit=15,
        border_mode=cv2.BORDER_REFLECT_101,
        p=0.5
    ),

    A.RandomBrightnessContrast(
        brightness_limit=0.15,
        contrast_limit=0.15,
        p=0.5
    ),

    A.HueSaturationValue(
        hue_shift_limit=5,
        sat_shift_limit=10,
        val_shift_limit=10,
        p=0.3
    ),

    A.CLAHE(
        clip_limit=2,
        p=0.3
    )

],
bbox_params=A.BboxParams(
    format="yolo",
    label_fields=["clases"],
    min_visibility=0.3
))

In [ ]:
def dibujar_boxes(img, labels, color=(0,255,0)):
    out = img.copy(); H, W = img.shape[:2]
    for c, xc, yc, w, h in labels:
        x1 = int((xc - w/2) * W); y1 = int((yc - h/2) * H)
        x2 = int((xc + w/2) * W); y2 = int((yc + h/2) * H)
        cv2.rectangle(out, (x1,y1), (x2,y2), color, 2)
        cv2.putText(out, CLASSES[c], (x1, y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return out

print("Función dibujar_boxes lista ✅")

In [ ]:
# ===== Comparación visual del Data Augmentation (corregida) =====
ejemplo_img = cv2.imread(train_imgs[0])
ejemplo_lbls = leer_labels(train_lbls[0])
bboxes = [[xc, yc, w, h] for _, xc, yc, w, h in ejemplo_lbls]
clases = [c for c, _, _, _, _ in ejemplo_lbls]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Original
show(dibujar_boxes(ejemplo_img, ejemplo_lbls), "ORIGINAL", ax=axes[0])

# 3 versiones aumentadas
for i in range(1, 4):
    r = transform(image=ejemplo_img, bboxes=bboxes, clases=clases)
    lbls_aug = [(int(c), *b) for c, b in zip(r["clases"], r["bboxes"])]   # int(c) aquí
    show(dibujar_boxes(r["image"], lbls_aug), f"AUMENTADA {i}", ax=axes[i])

plt.suptitle("Data Augmentation: original vs variaciones (con cajas ajustadas)", fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# ===== Renombrar la clase Mealybug -> Pseudococcidae =====
# Cambiar en la lista de clases
CLASSES = ["Pseudococcidae" if c == "Mealybug" else c for c in CLASSES]
CLASS_ID = {name: i for i, name in enumerate(CLASSES)}
print("Clases actualizadas:", CLASSES)

# Actualizar también el data.yaml (para que YOLO use el nombre nuevo al entrenar)
cfg = yaml.safe_load(open(DATA_YAML))
cfg["names"] = CLASSES
with open(DATA_YAML, "w") as f:
    yaml.safe_dump(cfg, f)
print("data.yaml actualizado ✅")

In [ ]:
# ===== PASO 4: Organizar dataset final para YOLO =====
FINAL = Path("dataset_final")
if FINAL.exists(): shutil.rmtree(FINAL)

# Train = las imágenes aumentadas que ya generamos
(FINAL / "train").mkdir(parents=True, exist_ok=True)
shutil.copytree(AUG_DIR / "images", FINAL / "train" / "images")
shutil.copytree(AUG_DIR / "labels", FINAL / "train" / "labels")

# Valid = las de validación originales (NO se aumentan, deben ser reales)
(FINAL / "valid" / "images").mkdir(parents=True, exist_ok=True)
(FINAL / "valid" / "labels").mkdir(parents=True, exist_ok=True)
for i, (img, lbl) in enumerate(zip(val_imgs, val_lbls)):
    shutil.copy(img, FINAL / "valid" / "images" / f"val{i:04d}.jpg")
    if os.path.exists(lbl):
        shutil.copy(lbl, FINAL / "valid" / "labels" / f"val{i:04d}.txt")
    else:
        open(FINAL / "valid" / "labels" / f"val{i:04d}.txt", "w").close()
...

In [ ]:
# ===== Forzar las 3 clases correctas =====
CLASSES = ["Grape", "Leaf", "Pseudococcidae"]
CLASS_ID = {name: i for i, name in enumerate(CLASSES)}
print("Clases corregidas:", CLASSES)

# Regenerar el data.yaml final con las clases correctas
YAML_FINAL = Path("dataset_final") / "data.yaml"
cfg = yaml.safe_load(open(YAML_FINAL))
cfg["names"] = CLASSES
cfg["nc"] = len(CLASSES)
with open(YAML_FINAL, "w") as f:
    yaml.safe_dump(cfg, f)
print("data.yaml final corregido ✅:", cfg["names"])

In [ ]:
# ===== Contar instancias reales por clase (no imágenes, sino cajas etiquetadas) =====
from collections import Counter
from pathlib import Path

def contar_instancias(carpeta_labels, nombres_clases):
    conteo = Counter()
    total_archivos = 0
    archivos_vacios = 0
    for lbl_path in Path(carpeta_labels).glob("*.txt"):
        total_archivos += 1
        lineas = open(lbl_path).read().strip().split("\n")
        if lineas == ['']:
            archivos_vacios += 1
            continue
        for linea in lineas:
            clase_id = int(float(linea.split()[0]))
            conteo[clase_id] += 1
    return conteo, total_archivos, archivos_vacios

print("=" * 50)
print("TRAIN")
print("=" * 50)
conteo_train, n_train, vacios_train = contar_instancias("dataset_final/train/labels", CLASSES)
for i, nombre in enumerate(CLASSES):
    print(f"  {nombre:15s}: {conteo_train.get(i, 0)} instancias")
print(f"  Archivos totales: {n_train} | Archivos sin ninguna caja: {vacios_train}")

print()
print("=" * 50)
print("VALID")
print("=" * 50)
conteo_val, n_val, vacios_val = contar_instancias("dataset_final/valid/labels", CLASSES)
for i, nombre in enumerate(CLASSES):
    print(f"  {nombre:15s}: {conteo_val.get(i, 0)} instancias")
print(f"  Archivos totales: {n_val} | Archivos sin ninguna caja: {vacios_val}")

# Grafico de barras para verlo de un vistazo
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1,2, figsize=(12,4))
ax[0].bar(CLASSES, [conteo_train.get(i,0) for i in range(len(CLASSES))], color=['#8B4513','#228B22','#800080'])
ax[0].set_title("Instancias por clase - TRAIN")
ax[1].bar(CLASSES, [conteo_val.get(i,0) for i in range(len(CLASSES))], color=['#8B4513','#228B22','#800080'])
ax[1].set_title("Instancias por clase - VALID")
plt.tight_layout(); plt.show()

In [ ]:
# ===== Corregir etiquetas: convertir clase decimal (2.0) a entero (2) =====
from collections import Counter
nums = Counter()
arreglados = 0

for lbl in (Path("dataset_final")/"train"/"labels").glob("*.txt"):
    lineas_nuevas = []
    cambio = False
    for line in open(lbl):
        p = line.split()
        if len(p) == 5:
            clase = int(float(p[0]))            # "2.0" -> 2.0 -> 2
            if p[0] != str(clase):
                cambio = True
            nums[clase] += 1
            lineas_nuevas.append(f"{clase} {p[1]} {p[2]} {p[3]} {p[4]}")
    open(lbl, "w").write("\n".join(lineas_nuevas))
    if cambio:
        arreglados += 1

print(f"Archivos corregidos: {arreglados}")
print("Números de clase ahora presentes:", dict(nums))
print("Deberían ser solo 0, 1, 2 (Grape, Leaf, Pseudococcidae)")

In [ ]:
# ===== Simplificar a detección de UNA sola clase: Pseudococcidae =====
import shutil

ID_PSEUDOCOCCIDAE = CLASSES.index("Pseudococcidae")  # debería ser 2

def filtrar_solo_pseudococcidae(carpeta_labels_origen, carpeta_labels_destino):
    Path(carpeta_labels_destino).mkdir(parents=True, exist_ok=True)
    for lbl_path in Path(carpeta_labels_origen).glob("*.txt"):
        lineas_nuevas = []
        for linea in open(lbl_path).read().strip().split("\n"):
            if linea == "":
                continue
            partes = linea.split()
            clase_id = int(float(partes[0]))
            if clase_id == ID_PSEUDOCOCCIDAE:
                # remapear a clase 0 (unica clase ahora)
                lineas_nuevas.append("0 " + " ".join(partes[1:]))
        # escribir el archivo (vacio si no tenia pseudococcidae, YOLO acepta esto)
        with open(Path(carpeta_labels_destino) / lbl_path.name, "w") as f:
            f.write("\n".join(lineas_nuevas))

filtrar_solo_pseudococcidae("dataset_final/train/labels", "dataset_final_v2/train/labels")
filtrar_solo_pseudococcidae("dataset_final/valid/labels", "dataset_final_v2/valid/labels")

# copiar las imagenes tal cual (no cambian)
shutil.copytree("dataset_final/train/images", "dataset_final_v2/train/images", dirs_exist_ok=True)
shutil.copytree("dataset_final/valid/images", "dataset_final_v2/valid/images", dirs_exist_ok=True)

# nuevo data.yaml con una sola clase
import yaml
cfg_v2 = {
    "train": "dataset_final_v2/train/images",
    "val": "dataset_final_v2/valid/images",
    "names": ["Pseudococcidae"],
    "nc": 1
}
with open("dataset_final_v2/data.yaml", "w") as f:
    yaml.safe_dump(cfg_v2, f)

print("Dataset v2 creado: 1 sola clase (Pseudococcidae)")

# Verificar el nuevo conteo
conteo_v2, n_v2, vacios_v2 = contar_instancias("dataset_final_v2/train/labels", ["Pseudococcidae"])
print("Instancias Pseudococcidae en v2:", conteo_v2.get(0,0), "de", n_v2, "archivos")

In [ ]:
# ===== Imagen base para las pruebas de preprocesamiento =====
SAMPLE = cv2.imread(train_imgs[0])
IMG_SIZE = 640
img_resized = cv2.resize(SAMPLE, (IMG_SIZE, IMG_SIZE))
gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
print("Imagen lista para preprocesamiento:", img_resized.shape)
plt.figure(figsize=(6,6)); show(img_resized, "Imagen base"); plt.show()

In [ ]:
# ===== 1. REDIMENSIONAMIENTO + 2. NORMALIZACIÓN =====
# Redimensionamiento (ya lo hicimos): a 640x640 para YOLO
print("Original:", SAMPLE.shape, "-> Redimensionada:", img_resized.shape)

# Normalización: pasar los píxeles de [0,255] a [0,1]
img_norm = img_resized.astype(np.float32) / 255.0
print("Rango antes:  min=%d max=%d" % (img_resized.min(), img_resized.max()))
print("Rango después: min=%.2f max=%.2f" % (img_norm.min(), img_norm.max()))

fig, ax = plt.subplots(1, 3, figsize=(18, 5))
show(SAMPLE, f"Original {SAMPLE.shape[1]}x{SAMPLE.shape[0]}", ax=ax[0])
show(img_resized, f"Redimensionada 640x640", ax=ax[1])
ax[2].hist(img_resized.ravel(), bins=50, color="gray", alpha=0.7)
ax[2].set_title("Histograma de intensidades")
ax[2].set_xlabel("Valor de píxel (0-255)")
plt.tight_layout(); plt.show()

In [ ]:
# ===== 3. ELIMINACIÓN DE RUIDO: comparación de 3 filtros =====
gb = cv2.GaussianBlur(img_resized, (5,5), 0)       # promedio ponderado gaussiano
mb = cv2.medianBlur(img_resized, 5)                # mediana del vecindario
bf = cv2.bilateralFilter(img_resized, 9, 75, 75)   # considera distancia + color

# Métrica objetiva: varianza del Laplaciano (a mayor valor, más bordes conservados)
def nitidez(im):
    g = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(g, cv2.CV_64F).var()

fig, ax = plt.subplots(1, 4, figsize=(20, 5))
for a, im, t in zip(ax, [img_resized, gb, mb, bf],
                    ["Original", "Gaussian Blur", "Median Blur", "Bilateral Filter"]):
    show(im, f"{t}\nnitidez bordes = {nitidez(im):.0f}", ax=a)
plt.tight_layout(); plt.show()

In [ ]:
# ===== 4. CLAHE: mejora de contraste adaptativo =====
# Se aplica sobre el canal L (luminosidad) de LAB para no dañar los colores
lab = cv2.cvtColor(img_resized, cv2.COLOR_BGR2LAB)
L, A, B = cv2.split(lab)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
L_clahe = clahe.apply(L)
img_clahe = cv2.cvtColor(cv2.merge([L_clahe, A, B]), cv2.COLOR_LAB2BGR)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
show(img_resized, "ANTES (original)", ax=ax[0])
show(img_clahe, "DESPUÉS (CLAHE)", ax=ax[1])
plt.tight_layout(); plt.show()

In [ ]:
# ===== 5. ECUALIZACIÓN DE HISTOGRAMA: global vs CLAHE =====
eq_global = cv2.equalizeHist(gray)          # ecualización clásica (toda la imagen)
eq_clahe = clahe.apply(gray)                # ecualización adaptativa (por regiones)

fig, ax = plt.subplots(2, 3, figsize=(16, 9))
# Fila 1: imágenes
show(gray, "Gris (original)", ax=ax[0,0])
show(eq_global, "Ecualización GLOBAL", ax=ax[0,1])
show(eq_clahe, "CLAHE (adaptativa)", ax=ax[0,2])
# Fila 2: histogramas
ax[1,0].hist(gray.ravel(), bins=256, color="gray");     ax[1,0].set_title("Histograma original")
ax[1,1].hist(eq_global.ravel(), bins=256, color="orange"); ax[1,1].set_title("Histograma global")
ax[1,2].hist(eq_clahe.ravel(), bins=256, color="teal");  ax[1,2].set_title("Histograma CLAHE")
plt.tight_layout(); plt.show()

In [ ]:
from pathlib import Path

pesos = list(Path("runs_vid").rglob("best.pt"))

print(pesos)

In [ ]:
# ===== 6. ESPACIOS DE COLOR: RGB / HSV / LAB =====
hsv = cv2.cvtColor(img_resized, cv2.COLOR_BGR2HSV)
lab2 = cv2.cvtColor(img_resized, cv2.COLOR_BGR2LAB)
rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(3, 3, figsize=(15, 12))
for i, canal in enumerate(["R (rojo)", "G (verde)", "B (azul)"]):
    show(rgb[:,:,i], f"RGB - {canal}", ax=ax[0,i])
for i, canal in enumerate(["H (tono)", "S (saturación)", "V (valor/brillo)"]):
    show(hsv[:,:,i], f"HSV - {canal}", ax=ax[1,i])
for i, canal in enumerate(["L (luminosidad)", "A (verde-rojo)", "B (azul-amarillo)"]):
    show(lab2[:,:,i], f"LAB - {canal}", ax=ax[2,i])
plt.suptitle("Comparación de espacios de color y sus canales", fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# ===== 7. CORRECCIÓN DE ILUMINACIÓN =====
# Modela la imagen como reflectancia × iluminación.
# Estima la iluminación (desenfoque fuerte) y la divide para dejar solo el objeto.
gray_f = gray.astype(np.float32) + 1
fondo = cv2.GaussianBlur(gray_f, (0,0), sigmaX=30)     # iluminación de fondo (baja frecuencia)
corr = cv2.normalize(gray_f / fondo, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

fig, ax = plt.subplots(1, 3, figsize=(16, 5))
show(gray, "Original (gris)", ax=ax[0])
show(fondo.astype(np.uint8), "Iluminación estimada", ax=ax[1])
show(corr, "Iluminación corregida", ax=ax[2])
plt.suptitle("Corrección de iluminación desigual", fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# ===== Elegir automáticamente una imagen SIMPLE para la demo de bordes =====
# (busca solo en train_imgs originales, NO en dataset_aug, para evitar rotaciones/bordes negros)
mejor_para_bordes = None
menor_densidad = 1.0
mejor_path = ""

for p in train_imgs[:80]:
    im = cv2.imread(p)
    if im is None:
        continue
    im = cv2.resize(im, (640, 640))
    g = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    e = cv2.Canny(g, 50, 150)
    densidad = (e > 0).mean()
    if 0.01 < densidad < menor_densidad:
        menor_densidad = densidad
        mejor_para_bordes = im
        mejor_path = p

print("Imagen elegida:", Path(mejor_path).name, f"(densidad: {menor_densidad*100:.1f}%)")
SAMPLE_BORDES = mejor_para_bordes
gray_bordes = cv2.cvtColor(SAMPLE_BORDES, cv2.COLOR_BGR2GRAY)
plt.figure(figsize=(6,6)); show(SAMPLE_BORDES, "Imagen para bordes"); plt.show()

In [ ]:
# Aumentar ligeramente el contraste local
clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8,8))
g = clahe.apply(gray_bordes)

# Reducir ruido
g = cv2.GaussianBlur(g, (3,3), 0)

In [ ]:
# ==========================================================
# Preparar imagen para detección de bordes
# ==========================================================

# Imagen original
img = SAMPLE_BORDES.copy()

# Convertir a LAB
lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

# Canal de luminancia
L, A, B = cv2.split(lab)

# CLAHE
clahe = cv2.createCLAHE(
    clipLimit=2.5,
    tileGridSize=(8,8)
)

L = clahe.apply(L)

# Suavizado ligero
L = cv2.GaussianBlur(L,(3,3),0)

In [ ]:
# ===== 8. DETECCIÓN DE BORDES: Sobel / Laplaciano / Canny =====

g = cv2.bilateralFilter(gray_bordes, 9, 50, 50)

def limpiar_por_tamano(mask_bin, area_min=15):
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask_bin, connectivity=8)
    limpio = np.zeros_like(mask_bin)
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] >= area_min:
            limpio[labels == i] = 255
    return limpio

# --- Sobel ---
sobel = cv2.magnitude(cv2.Sobel(g, cv2.CV_64F, 1, 0, ksize=3),
                      cv2.Sobel(g, cv2.CV_64F, 0, 1, ksize=3))
sobel = cv2.normalize(sobel, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
_, sobel = cv2.threshold(sobel, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
sobel = limpiar_por_tamano(sobel, area_min=15)
sobel = 255 - sobel

# --- Laplaciano ---
g_lap = cv2.GaussianBlur(g, (7,7), 7/3)
lap = cv2.convertScaleAbs(cv2.Laplacian(g_lap, cv2.CV_64F, ksize=3))
lap = cv2.normalize(lap, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
umbral_lap = np.percentile(lap, 97)
_, lap = cv2.threshold(lap, umbral_lap, 255, cv2.THRESH_BINARY)
lap = limpiar_por_tamano(lap, area_min=15)
lap = 255 - lap

# --- Canny ---
canny = cv2.Canny(g, 20, 60)
canny = limpiar_por_tamano(canny, area_min=15)
canny = 255 - canny

fig, ax = plt.subplots(1, 4, figsize=(20, 5))
show(gray_bordes, "Original (gris)", ax=ax[0])
show(sobel, "Sobel", ax=ax[1])
show(lap, "Laplaciano", ax=ax[2])
show(canny, "Canny", ax=ax[3])
plt.tight_layout(); plt.show()

In [ ]:
# ===== Buscar una imagen fácil de segmentar (mealybug bien contrastado) =====
mejor_img = None
mejor_score = 0
mejor_path = ""

# Revisar las primeras 100 imágenes de train
for p in train_imgs[:100]:
    im = cv2.imread(p)
    if im is None:
        continue
    im = cv2.resize(im, (640, 640))
    hsv = cv2.cvtColor(im, cv2.COLOR_BGR2HSV)
    # Buscar zonas blancas-algodonosas (mealybug): baja saturación, alto brillo
    mask = cv2.inRange(hsv, (0, 0, 180), (180, 60, 255))
    blancos = cv2.countNonZero(mask)
    # Score: que tenga mealybug visible pero no demasiado fondo blanco
    porcentaje = blancos / (640*640)
    if 0.01 < porcentaje < 0.12 and blancos > mejor_score:
        mejor_score = blancos
        mejor_img = im
        mejor_path = p

if mejor_img is not None:
    print("Imagen elegida:", Path(mejor_path).name)
    SEG_IMG = mejor_img
    SEG_GRAY = cv2.cvtColor(SEG_IMG, cv2.COLOR_BGR2GRAY)
    plt.figure(figsize=(7,7)); show(SEG_IMG, "Imagen elegida para segmentación"); plt.show()
else:
    print("No se encontró una imagen ideal, usando la base")
    SEG_IMG = img_resized
    SEG_GRAY = gray

In [ ]:
# ===== SEGMENTACIÓN: aislar mealybug por saturación (HSV) =====
hsv = cv2.cvtColor(SEG_IMG, cv2.COLOR_BGR2HSV)
S = hsv[:,:,1]   # canal saturación: el mealybug (grisáceo) tiene baja saturación
V = hsv[:,:,2]   # canal brillo: lo necesitamos para excluir reflejos especulares

# Umbral base por saturación
_, th_glob = cv2.threshold(S, 60, 255, cv2.THRESH_BINARY_INV)
_, th_otsu = cv2.threshold(S, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

# CORRECCIÓN 1: excluir brillo especular (reflejos de luz también tienen baja
# saturación, igual que la cochinilla, pero además son extremadamente brillantes)
th_glob[V > 240] = 0
th_otsu[V > 240] = 0

# CORRECCIÓN 2: limpiar ruido "sal y pimienta" disperso por el fondo
th_glob = cv2.medianBlur(th_glob, 7)
th_otsu = cv2.medianBlur(th_otsu, 7)

# CORRECCIÓN 3: adaptativa con bloque más grande (21 era demasiado chico,
# muy sensible al ruido pixel a pixel) + limpieza
th_adap = cv2.adaptiveThreshold(S, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                cv2.THRESH_BINARY_INV, 51, 6)
th_adap = cv2.medianBlur(th_adap, 7)

fig, ax = plt.subplots(1, 4, figsize=(20, 5))
show(SEG_IMG, "Original", ax=ax[0])
show(th_glob, "Global (saturación)", ax=ax[1])
show(th_otsu, "Otsu (saturación)", ax=ax[2])
show(th_adap, "Adaptativa", ax=ax[3])
plt.suptitle("Segmentación por saturación (HSV)", fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# ===== SEGMENTACIÓN: Operaciones morfológicas (sobre la máscara de Otsu) =====
mask = th_otsu   # la máscara buena de la celda anterior
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))

ero = cv2.erode(mask, kernel, iterations=1)
dil = cv2.dilate(mask, kernel, iterations=1)
opening = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)   # limpia puntitos
closing = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)  # rellena huecos

fig, ax = plt.subplots(1, 5, figsize=(20, 5))
for a, im, t in zip(ax, [mask, ero, dil, opening, closing],
                    ["Otsu (base)", "Erosión", "Dilatación", "Opening", "Closing"]):
    show(im, t, ax=a)
plt.suptitle("Operaciones morfológicas", fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# ===== EXTRACCIÓN DE CARACTERÍSTICAS =====
# Usamos la máscara limpia del mealybug (opening quita ruido)
mask_limpia = cv2.morphologyEx(th_otsu, cv2.MORPH_OPEN,
                               cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5)))

def extraer_caracteristicas(img_bgr, mask):
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None, None
    c = max(cnts, key=cv2.contourArea)          # el contorno más grande (grupo de plaga)

    area = cv2.contourArea(c)
    perim = cv2.arcLength(c, True)
    x, y, w, h = cv2.boundingRect(c)
    hull = cv2.convexHull(c)
    area_hull = cv2.contourArea(hull)
    circularidad = 4*np.pi*area/(perim**2 + 1e-6)     # 1 = círculo perfecto
    solidez = area/(area_hull + 1e-6)                 # compacidad
    hu = cv2.HuMoments(cv2.moments(c)).flatten()      # 7 momentos invariantes
    color_prom = cv2.mean(img_bgr, mask=mask)[:3]     # color promedio (B,G,R)
    hist = [cv2.calcHist([img_bgr],[i],mask,[32],[0,256]).flatten() for i in range(3)]

    # Textura (LBP y GLCM)
    gray_r = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    glcm = graycomatrix(gray_r, [1], [0], 256, symmetric=True, normed=True)
    textura = {p: float(graycoprops(glcm, p)[0,0])
               for p in ["contrast","homogeneity","energy","correlation"]}

    feats = dict(area=area, perimetro=round(perim,1), bbox=(x,y,w,h),
                 circularidad=round(circularidad,3), solidez=round(solidez,3),
                 color_prom_BGR=tuple(round(v,1) for v in color_prom),
                 hu_moments=hu, textura=textura)
    return feats, (c, hull, hist)

feats, extra = extraer_caracteristicas(SEG_IMG, mask_limpia)

if feats:
    print("="*45)
    print("     CARACTERÍSTICAS EXTRAÍDAS")
    print("="*45)
    for k, v in feats.items():
        if k == "hu_moments":
            print(f"  Momentos de Hu: {[round(float(x),4) for x in v]}")
        elif k == "textura":
            print(f"  Textura (GLCM): {v}")
        else:
            print(f"  {k}: {v}")
else:
    print("No se detectó contorno. Revisa la máscara.")

In [ ]:
# ===== EXTRACCIÓN DE CARACTERÍSTICAS (solo el mealybug, sin ruido) =====
# Limpiar: opening fuerte + quedarse solo con contornos grandes
mask_limpia = cv2.morphologyEx(th_otsu, cv2.MORPH_OPEN,
                               cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7,7)))

# Filtrar por tamaño: eliminar manchas pequeñas (ruido/brillos), dejar solo la plaga
cnts, _ = cv2.findContours(mask_limpia, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
mask_final = np.zeros_like(mask_limpia)
area_max = max([cv2.contourArea(c) for c in cnts]) if cnts else 0
for c in cnts:
    if cv2.contourArea(c) > area_max * 0.15:   # solo manchas >15% de la más grande
        cv2.drawContours(mask_final, [c], -1, 255, -1)

# Mostrar la limpieza
fig, ax = plt.subplots(1, 3, figsize=(16, 5))
show(SEG_IMG, "Original", ax=ax[0])
show(th_otsu, "Máscara con ruido", ax=ax[1])
show(mask_final, "Solo el mealybug (filtrado por tamaño)", ax=ax[2])
plt.tight_layout(); plt.show()

def extraer_caracteristicas(img_bgr, mask):
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None, None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    perim = cv2.arcLength(c, True)
    x, y, w, h = cv2.boundingRect(c)
    hull = cv2.convexHull(c)
    circularidad = 4*np.pi*area/(perim**2 + 1e-6)
    solidez = area/(cv2.contourArea(hull) + 1e-6)
    hu = cv2.HuMoments(cv2.moments(c)).flatten()
    color_prom = cv2.mean(img_bgr, mask=mask)[:3]
    hist = [cv2.calcHist([img_bgr],[i],mask,[32],[0,256]).flatten() for i in range(3)]
    gray_r = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    glcm = graycomatrix(gray_r, [1], [0], 256, symmetric=True, normed=True)
    textura = {p: float(graycoprops(glcm, p)[0,0])
               for p in ["contrast","homogeneity","energy","correlation"]}
    feats = dict(area=area, perimetro=round(perim,1), bbox=(x,y,w,h),
                 circularidad=round(circularidad,3), solidez=round(solidez,3),
                 color_prom_BGR=tuple(round(v,1) for v in color_prom),
                 hu_moments=hu, textura=textura)
    return feats, (c, hull, hist)

feats, extra = extraer_caracteristicas(SEG_IMG, mask_final)
if feats:
    print("="*45)
    print("     CARACTERÍSTICAS DEL MEALYBUG")
    print("="*45)
    for k, v in feats.items():
        if k == "hu_moments":
            print(f"  Momentos de Hu: {[round(float(x),4) for x in v]}")
        elif k == "textura":
            print(f"  Textura (GLCM): {v}")
        else:
            print(f"  {k}: {v}")

In [ ]:
from ultralytics import YOLO
import torch
print("Ultralytics OK ✅")
print("¿GPU disponible?:", "SÍ 🚀" if torch.cuda.is_available() else "No, usará CPU")
if torch.cuda.is_available():
    print("GPU detectada:", torch.cuda.get_device_name(0))

In [ ]:
import sys
# Desinstalar la versión CPU y instalar la versión con CUDA (para RTX 3050)
!{sys.executable} -m pip uninstall -y torch torchvision
!{sys.executable} -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

In [ ]:
import torch
print("Versión de PyTorch:", torch.__version__)
print("¿GPU disponible?:", "SÍ 🚀" if torch.cuda.is_available() else "No, usará CPU")
if torch.cuda.is_available():
    print("GPU detectada:", torch.cuda.get_device_name(0))

In [ ]:
# ===== Recargar lo mínimo para entrenar =====
from pathlib import Path
import yaml
from ultralytics import YOLO

YAML_FINAL = Path("dataset_final") / "data.yaml"
cfg = yaml.safe_load(open(YAML_FINAL))
CLASSES = cfg["names"]
IMG_SIZE = 640
print("Clases:", CLASSES)
print("Config lista:", YAML_FINAL.resolve())

In [ ]:
from pathlib import Path
import yaml
from ultralytics import YOLO
YAML_FINAL = Path("dataset_final") / "data.yaml"
print("Config lista:", YAML_FINAL.exists())

In [ ]:
model = YOLO("yolo11n.pt")
results = model.train(
    data=str(YAML_FINAL),
    epochs=60,
    imgsz=640,
    batch=8,
    workers=0,        # 👈 CLAVE: evita el cuelgue en Windows
    patience=15,
    device=0,
    project="runs_vid",
    name="yolo11_pseudococcidae2",
    plots=True,
)
print("✅ Terminado:", results.save_dir)

In [ ]:
from IPython.display import Image, display
from pathlib import Path

RUN_DIR = Path(r"C:\Users\PAULA\runs\detect\runs_vid\yolo11_pseudococcidae2")

# Mostrar las gráficas que YOLO generó
for nombre in ["results.png", "confusion_matrix.png", "PR_curve.png", "F1_curve.png"]:
    p = RUN_DIR / nombre
    if p.exists():
        print("📊", nombre)
        display(Image(str(p)))

In [ ]:
from ultralytics import YOLO
from pathlib import Path

model = YOLO("yolo11s.pt")   # modelo SMALL: más capacidad que el nano
results = model.train(
    data=str(Path("dataset_final") / "data.yaml"),
    epochs=100,
    imgsz=640,
    batch=8,
    workers=0,
    patience=25,
    device=0,
    project="runs_vid",
    name="yolo11_mejorado",
    plots=True,
)
print("✅ Terminado:", results.save_dir)

In [ ]:
from ultralytics import YOLO
from pathlib import Path

model = YOLO("yolo11s.pt")   # modelo SMALL: más capacidad que el nano
results = model.train(
    data=str(Path("dataset_final") / "data.yaml"),
    epochs=100,
    imgsz=640,
    batch=8,
    workers=0,
    patience=25,
    device=0,
    project="runs_vid",
    name="yolo11_mejorado",
    plots=True,
)
print("✅ Terminado:", results.save_dir)

In [ ]:
from pathlib import Path

print("Carpeta actual de Jupyter:", Path.cwd())
print()
print("dataset_final existe:", Path("dataset_final").exists())
print("dataset_final/data.yaml existe:", Path("dataset_final/data.yaml").exists())
print()
print("Carpetas de entrenamientos ya hechos (runs_vid):")
if Path("runs_vid").exists():
    for p in Path("runs_vid").rglob("best.pt"):
        print("  ✅ modelo entrenado encontrado:", p)
else:
    print("  ❌ no existe runs_vid en esta carpeta")

In [ ]:
from pathlib import Path
print("Carpeta actual:", Path.cwd())
print("dataset_final existe:", Path("dataset_final").exists())

In [ ]:
ID_PSEUDOCOCCIDAE = CLASSES.index("Pseudococcidae")

def filtrar_solo_pseudococcidae(carpeta_labels_origen, carpeta_labels_destino):
    Path(carpeta_labels_destino).mkdir(parents=True, exist_ok=True)
    for lbl_path in Path(carpeta_labels_origen).glob("*.txt"):
        lineas_nuevas = []
        for linea in open(lbl_path).read().strip().split("\n"):
            if linea == "":
                continue
            partes = linea.split()
            clase_id = int(float(partes[0]))
            if clase_id == ID_PSEUDOCOCCIDAE:
                lineas_nuevas.append("0 " + " ".join(partes[1:]))
        with open(Path(carpeta_labels_destino) / lbl_path.name, "w") as f:
            f.write("\n".join(lineas_nuevas))

filtrar_solo_pseudococcidae("dataset_final/train/labels", "dataset_final_v2/train/labels")
filtrar_solo_pseudococcidae("dataset_final/valid/labels", "dataset_final_v2/valid/labels")

shutil.copytree("dataset_final/train/images", "dataset_final_v2/train/images", dirs_exist_ok=True)
shutil.copytree("dataset_final/valid/images", "dataset_final_v2/valid/images", dirs_exist_ok=True)

cfg_v2 = {
    "train": "train/images",
    "val": "valid/images",
    "names": ["Pseudococcidae"],
    "nc": 1
}
with open("dataset_final_v2/data.yaml", "w") as f:
    yaml.safe_dump(cfg_v2, f)

print("Dataset v2 creado ✅")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
results = model.train(
    data="dataset_final_v2/data.yaml",
    epochs=80,
    imgsz=640,
    batch=8,
    project="runs_vid",
    name="yolo11_pseudococcidae_v2"
)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
results = model.train(
    data="dataset_final_v2/data.yaml",
    epochs=100,
    patience=15,
    imgsz=640,
    batch=8,
    project="runs_vid",
    name="yolo11_pseudococcidae_v3"
)

In [ ]:
from ultralytics import YOLO

model = YOLO("runs/detect/runs_vid/yolo11_pseudococcidae_v3/weights/best.pt")

metrics = model.val(data="dataset_final_v2/data.yaml")

print("=" * 50)
print("RESULTADOS EN PORCENTAJE")
print("=" * 50)
print(f"Precision   : {metrics.box.mp * 100:.2f}%")
print(f"Recall      : {metrics.box.mr * 100:.2f}%")
print(f"mAP@0.5     : {metrics.box.map50 * 100:.2f}%")
print(f"mAP@0.5:0.95: {metrics.box.map * 100:.2f}%")

p = metrics.box.mp
r = metrics.box.mr
f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0
print(f"F1-Score    : {f1 * 100:.2f}%")

In [ ]:
model = YOLO("yolo11s.pt")
results = model.train(
    data="dataset_final_v2/data.yaml",
    epochs=100,
    patience=20,        # un poco más de paciencia, ya que s es más lento en converger
    imgsz=640,
    batch=8,
    project="runs_vid",
    name="yolo11_pseudococcidae_v4"
)

In [ ]:
import cv2
import random
import matplotlib.pyplot as plt
from pathlib import Path

# Cambia la ruta si es diferente
img_dir = Path("dataset_final/images/train")
label_dir = Path("dataset_final/labels/train")

imagenes = list(img_dir.glob("*.*"))

# Mostrar 10 imágenes aleatorias
muestras = random.sample(imagenes, 10)

plt.figure(figsize=(15,20))

for i, img_path in enumerate(muestras):

    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]

    label_path = label_dir / (img_path.stem + ".txt")

    if label_path.exists():

        with open(label_path) as f:

            for linea in f:

                c, xc, yc, bw, bh = map(float, linea.split())

                x1 = int((xc - bw/2) * w)
                y1 = int((yc - bh/2) * h)
                x2 = int((xc + bw/2) * w)
                y2 = int((yc + bh/2) * h)

                cv2.rectangle(img,(x1,y1),(x2,y2),(0,255,0),2)

    plt.subplot(5,2,i+1)
    plt.imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(img_path.name)

plt.tight_layout()
plt.show()

In [ ]:
import random
import matplotlib.pyplot as plt

print("Ruta que está buscando:", img_dir)
print("Existe la carpeta?:", img_dir.exists())

imagenes = list(img_dir.glob("*.*"))
print("Cantidad de imágenes encontradas:", len(imagenes))
print("Primeros archivos:", imagenes[:5])

# Mostrar hasta 10 imágenes aleatorias (o menos, si no hay suficientes)
n_muestras = min(10, len(imagenes))
muestras = random.sample(imagenes, n_muestras)

plt.figure(figsize=(15,20))
for i, img_path in enumerate(muestras):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(5, 2, i+1)
    plt.imshow(img)
    plt.title(img_path.name, fontsize=9)
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
import random
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

img_dir = Path("dataset_final_v2") / "train" / "images"

print("Ruta que está buscando:", img_dir)
print("Existe la carpeta?:", img_dir.exists())

imagenes = list(img_dir.glob("*.*"))
print("Cantidad de imágenes encontradas:", len(imagenes))
print("Primeros archivos:", imagenes[:5])

n_muestras = min(10, len(imagenes))
muestras = random.sample(imagenes, n_muestras)

plt.figure(figsize=(15,20))
for i, img_path in enumerate(muestras):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(5, 2, i+1)
    plt.imshow(img)
    plt.title(img_path.name, fontsize=9)
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
import random
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

img_dir = Path("dataset_final_v2") / "train" / "images"
lbl_dir = Path("dataset_final_v2") / "train" / "labels"

imagenes = list(img_dir.glob("*.*"))
n_muestras = min(10, len(imagenes))
muestras = random.sample(imagenes, n_muestras)

plt.figure(figsize=(15,20))
for i, img_path in enumerate(muestras):
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]
    lbl_path = lbl_dir / (img_path.stem + ".txt")
    if lbl_path.exists():
        for linea in open(lbl_path).read().strip().split("\n"):
            if linea == "": continue
            _, xc, yc, bw, bh = map(float, linea.split())
            x1, y1 = int((xc-bw/2)*w), int((yc-bh/2)*h)
            x2, y2 = int((xc+bw/2)*w), int((yc+bh/2)*h)
            cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 3)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(5, 2, i+1)
    plt.imshow(img)
    plt.title(img_path.name, fontsize=9)
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
%matplotlib inline
import random
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

img_dir = Path("dataset_final_v2") / "train" / "images"
lbl_dir = Path("dataset_final_v2") / "train" / "labels"

imagenes = list(img_dir.glob("*.*"))
n_muestras = min(10, len(imagenes))
muestras = random.sample(imagenes, n_muestras)

plt.figure(figsize=(15,20))
for i, img_path in enumerate(muestras):
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]
    lbl_path = lbl_dir / (img_path.stem + ".txt")
    if lbl_path.exists():
        for linea in open(lbl_path).read().strip().split("\n"):
            if linea == "": continue
            _, xc, yc, bw, bh = map(float, linea.split())
            x1, y1 = int((xc-bw/2)*w), int((yc-bh/2)*h)
            x2, y2 = int((xc+bw/2)*w), int((yc+bh/2)*h)
            cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 3)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(5, 2, i+1)
    plt.imshow(img)
    plt.title(img_path.name, fontsize=9)
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
%matplotlib inline
import random
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

img_dir = Path("dataset_final_v2") / "train" / "images"
lbl_dir = Path("dataset_final_v2") / "train" / "labels"

imagenes = list(img_dir.glob("*.*"))
random.seed(42)  # para que siempre salgan las mismas 30, y puedas anotar cuáles están mal
n_muestras = min(30, len(imagenes))
muestras = random.sample(imagenes, n_muestras)

plt.figure(figsize=(18, 36))
for i, img_path in enumerate(muestras):
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]
    lbl_path = lbl_dir / (img_path.stem + ".txt")
    n_cajas = 0
    if lbl_path.exists():
        lineas = [l for l in open(lbl_path).read().strip().split("\n") if l != ""]
        n_cajas = len(lineas)
        for linea in lineas:
            _, xc, yc, bw, bh = map(float, linea.split())
            x1, y1 = int((xc-bw/2)*w), int((yc-bh/2)*h)
            x2, y2 = int((xc+bw/2)*w), int((yc+bh/2)*h)
            cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 3)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(6, 5, i+1)
    plt.imshow(img)
    plt.title(f"[{i}] {img_path.name}\n({n_cajas} cajas)", fontsize=8)
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
sospechosas = []

for lbl_path in lbl_dir.glob("*.txt"):
    lineas = [l for l in open(lbl_path).read().strip().split("\n") if l != ""]
    n_cajas = len(lineas)

    razones = []
    if n_cajas > 6:
        razones.append(f"muchas cajas ({n_cajas})")

    for linea in lineas:
        _, xc, yc, bw, bh = map(float, linea.split())
        area = bw * bh
        if area > 0.7:
            razones.append("caja gigante (>70% de la imagen)")
        if area < 0.001:
            razones.append("caja microscópica (<0.1% de la imagen)")

    if razones:
        img_path = img_dir / (lbl_path.stem + ".jpg")
        sospechosas.append((img_path, list(set(razones))))

print(f"Total de imágenes sospechosas encontradas: {len(sospechosas)} de {len(list(lbl_dir.glob('*.txt')))}")
print()
for path, razones in sospechosas[:20]:  # muestra las primeras 20
    print(f"{path.name}: {razones}")

In [ ]:
# Cuántas imágenes BASE distintas (sin contar variaciones _a0, _a1) están generando la alerta
import re

base_names = set()
for lbl_path in lbl_dir.glob("*.txt"):
    lineas = [l for l in open(lbl_path).read().strip().split("\n") if l != ""]
    n_cajas = len(lineas)
    tiene_micro = any(float(l.split()[3]) * float(l.split()[4]) < 0.001 for l in lineas)
    if n_cajas > 6 or tiene_micro:
        base = re.sub(r'_a\d+$', '', lbl_path.stem)  # quita el sufijo _a0, _a1 si existe
        base_names.add(base)

print(f"Imágenes BASE distintas con alerta: {len(base_names)}")

In [ ]:
# Visualizar específicamente las que tienen cajas MICROSCÓPICAS, para ver si son reales o ruido
micro_sospechosas = []
for lbl_path in lbl_dir.glob("*.txt"):
    lineas = [l for l in open(lbl_path).read().strip().split("\n") if l != ""]
    for linea in lineas:
        partes = linea.split()
        area = float(partes[3]) * float(partes[4])
        if area < 0.001:
            micro_sospechosas.append(lbl_path)
            break

print(f"Total con al menos 1 caja microscópica: {len(micro_sospechosas)}")

# Mostrar 9 ejemplos
muestra_micro = random.sample(micro_sospechosas, min(9, len(micro_sospechosas)))
plt.figure(figsize=(15, 15))
for i, lbl_path in enumerate(muestra_micro):
    img_path = img_dir / (lbl_path.stem + ".jpg")
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]
    for linea in open(lbl_path).read().strip().split("\n"):
        if linea == "": continue
        _, xc, yc, bw, bh = map(float, linea.split())
        x1, y1 = int((xc-bw/2)*w), int((yc-bh/2)*h)
        x2, y2 = int((xc+bw/2)*w), int((yc+bh/2)*h)
        color = (255,0,0) if (bw*bh) < 0.001 else (0,255,0)  # rojo = microscópica, verde = normal
        cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(3,3,i+1)
    plt.imshow(img)
    plt.title(lbl_path.stem, fontsize=8)
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Crear una versión v3 del dataset SIN cajas microscópicas (área < 0.001)
import shutil

def filtrar_cajas_micro(carpeta_labels_origen, carpeta_labels_destino, area_min=0.001):
    Path(carpeta_labels_destino).mkdir(parents=True, exist_ok=True)
    eliminadas = 0
    for lbl_path in Path(carpeta_labels_origen).glob("*.txt"):
        lineas_nuevas = []
        for linea in open(lbl_path).read().strip().split("\n"):
            if linea == "": continue
            partes = linea.split()
            area = float(partes[3]) * float(partes[4])
            if area >= area_min:
                lineas_nuevas.append(linea)
            else:
                eliminadas += 1
        with open(Path(carpeta_labels_destino) / lbl_path.name, "w") as f:
            f.write("\n".join(lineas_nuevas))
    print(f"Cajas microscópicas eliminadas: {eliminadas}")

filtrar_cajas_micro("dataset_final_v2/train/labels", "dataset_final_v3/train/labels")
filtrar_cajas_micro("dataset_final_v2/valid/labels", "dataset_final_v3/valid/labels")

shutil.copytree("dataset_final_v2/train/images", "dataset_final_v3/train/images", dirs_exist_ok=True)
shutil.copytree("dataset_final_v2/valid/images", "dataset_final_v3/valid/images", dirs_exist_ok=True)

import yaml
cfg_v3 = {"train": "train/images", "val": "valid/images", "names": ["Pseudococcidae"], "nc": 1}
with open("dataset_final_v3/data.yaml", "w") as f:
    yaml.safe_dump(cfg_v3, f)

print("dataset_final_v3 creado (sin cajas microscópicas) ✅")

In [ ]:
from collections import Counter
from pathlib import Path

def contar_instancias(carpeta_labels, nombres_clases):
    conteo = Counter()
    total_archivos = 0
    archivos_vacios = 0
    for lbl_path in Path(carpeta_labels).glob("*.txt"):
        total_archivos += 1
        lineas = open(lbl_path).read().strip().split("\n")
        if lineas == ['']:
            archivos_vacios += 1
            continue
        for linea in lineas:
            clase_id = int(float(linea.split()[0]))
            conteo[clase_id] += 1
    return conteo, total_archivos, archivos_vacios

In [ ]:
conteo_v3, n_v3, vacios_v3 = contar_instancias("dataset_final_v3/train/labels", ["Pseudococcidae"])
print(f"TRAIN v3 -> Instancias Pseudococcidae: {conteo_v3.get(0,0)}")
print(f"TRAIN v3 -> Total archivos: {n_v3} | Sin ninguna caja: {vacios_v3}")
print()

conteo_v3_val, n_v3_val, vacios_v3_val = contar_instancias("dataset_final_v3/valid/labels", ["Pseudococcidae"])
print(f"VALID v3 -> Instancias Pseudococcidae: {conteo_v3_val.get(0,0)}")
print(f"VALID v3 -> Total archivos: {n_v3_val} | Sin ninguna caja: {vacios_v3_val}")

In [ ]:
from pathlib import Path

v4_path = Path("runs/detect/runs_vid/yolo11_pseudococcidae_v4/weights/best.pt")
if v4_path.exists():
    print("✅ v4 ya terminó — el archivo best.pt existe")
    print("Última modificación:", v4_path.stat().st_mtime)
else:
    print("❌ v4 todavía no ha generado best.pt — probablemente sigue entrenando")

In [ ]:
from ultralytics import YOLO

model_v4 = YOLO("runs/detect/runs_vid/yolo11_pseudococcidae_v4/weights/best.pt")
metrics_v4 = model_v4.val(data="dataset_final_v2/data.yaml")

print("=" * 50)
print("RESULTADOS v4 (yolo11s, con cajas microscópicas)")
print("=" * 50)
print(f"Precision   : {metrics_v4.box.mp * 100:.2f}%")
print(f"Recall      : {metrics_v4.box.mr * 100:.2f}%")
print(f"mAP@0.5     : {metrics_v4.box.map50 * 100:.2f}%")
print(f"mAP@0.5:0.95: {metrics_v4.box.map * 100:.2f}%")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")   # volvemos al nano, que rindió mejor
results = model.train(
    data="dataset_final_v3/data.yaml",
    epochs=100,
    patience=20,
    imgsz=640,
    batch=8,
    project="runs_vid",
    name="yolo11_pseudococcidae_v5"
)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
results = model.train(
    data="dataset_final_v3/data.yaml",
    epochs=100,
    patience=20,
    imgsz=640,
    batch=8,
    workers=2,          # <-- NUEVO: bajado de 8 (default) a 2
    project="runs_vid",
    name="yolo11_pseudococcidae_v5"
)

In [ ]:
from pathlib import Path

for p in sorted((Path.home() / "runs").rglob("best.pt")):
    print(p)

In [ ]:
from ultralytics import YOLO

pesos = {
    "v3 (11n, sucio)":  r"C:\Users\PAULA\runs\detect\runs_vid\yolo11_pseudococcidae_v3\weights\best.pt",
    "v4 (11s, sucio)":  r"C:\Users\PAULA\runs\detect\runs_vid\yolo11_pseudococcidae_v4\weights\best.pt",
    "v5 (11n, limpio)": r"C:\Users\PAULA\runs\detect\runs_vid\yolo11_pseudococcidae_v5-3\weights\best.pt",
}

for valid_set in ["dataset_final_v2", "dataset_final_v3"]:
    print(f"\n===== VALID DE {valid_set.upper()} =====")
    for etiqueta, w in pesos.items():
        r = YOLO(w).val(
            data=f"{valid_set}/data.yaml",
            imgsz=640, batch=8, workers=2, verbose=False, plots=False,
            project="comparacion", name=f"{etiqueta[:2]}_en_{valid_set[-2:]}"
        )
        p, rc = r.box.mp, r.box.mr
        f1 = 2 * p * rc / (p + rc + 1e-9)
        print(f"  {etiqueta:18s} P={p:.4f}  R={rc:.4f}  mAP50={r.box.map50:.4f}  "
              f"mAP50-95={r.box.map:.4f}  F1={f1:.4f}")

In [ ]:
# =====================================================================
# CELDA FINAL — RESULTADOS DEL MODELO SELECCIONADO
# Detección de Pseudococcidae en vid con YOLO11
# =====================================================================
import pandas as pd
from pathlib import Path
from IPython.display import display, Image, Markdown

RUTA_V3 = Path(r"C:\Users\PAULA\runs\detect\runs_vid\yolo11_pseudococcidae_v3")

# --- Validación en vivo sobre el conjunto original (métrica primaria) ---
try:
    from ultralytics import YOLO
    r = YOLO(str(RUTA_V3 / "weights" / "best.pt")).val(
        data="dataset_final_v2/data.yaml",
        imgsz=640, batch=8, workers=2, verbose=False, plots=False,
        project="presentacion", name="modelo_final", exist_ok=True
    )
    P, R = r.box.mp, r.box.mr
    mAP50, mAP5095 = r.box.map50, r.box.map
except Exception as e:
    print(f"[Aviso] No se pudo revalidar ({e}). Se usan los valores registrados.")
    P, R, mAP50, mAP5095 = 0.6284, 0.4213, 0.4571, 0.2277

F1 = 2 * P * R / (P + R)

display(Markdown("## Modelo final: YOLO11n — 1 clase (Pseudococcidae)"))
display(Markdown(
    "*Entrenado sobre `dataset_final_v2` (2331 img. train / 68 valid). "
    "Métricas medidas sobre el conjunto de validación original, sin modificar.*"
))

display(pd.DataFrame({
    "Métrica": ["Precision", "Recall", "mAP@0.5", "mAP@0.5:0.95", "F1-Score"],
    "Valor":   [f"{P:.2%}", f"{R:.2%}", f"{mAP50:.2%}", f"{mAP5095:.2%}", f"{F1:.2%}"]
}).style.hide(axis="index").set_caption("Desempeño del modelo final"))

# --- Comparación de las tres configuraciones evaluadas ---
display(Markdown("## Comparación de configuraciones (mismo conjunto de validación)"))

comp = pd.DataFrame([
    ["v3 — YOLO11n", "dataset_final_v2", "62.84%", "42.13%", "45.71%", "22.77%", "50.44%"],
    ["v4 — YOLO11s", "dataset_final_v2", "54.55%", "40.28%", "41.51%", "20.54%", "46.34%"],
    ["v5 — YOLO11n", "dataset_final_v3", "61.45%", "39.85%", "45.05%", "21.87%", "48.35%"],
], columns=["Configuración", "Dataset entren.", "Precision", "Recall",
            "mAP@0.5", "mAP@0.5:0.95", "F1"])

display(comp.style.hide(axis="index").apply(
    lambda x: ["font-weight: bold; background-color: #e8f4ea"] * len(x)
              if x.name == 0 else [""] * len(x), axis=1
).set_caption("Ablación: efecto de la arquitectura y de la limpieza de anotaciones"))

display(Markdown(
    "**Hallazgo.** Aumentar la capacidad del modelo (v4, YOLO11s: 9.4M vs 2.6M parámetros) "
    "degradó el desempeño en todas las métricas. La limpieza de anotaciones microscópicas "
    "(v5) no produjo un efecto detectable con el tamaño de validación disponible. "
    "Ambos resultados señalan que la limitación principal es la **cantidad y diversidad "
    "del conjunto de datos**, no la arquitectura ni la calidad del etiquetado.\n\n"
    "**Nota metodológica.** Evaluar el mismo modelo v3 sobre el conjunto filtrado elevaba "
    "su mAP@0.5 de 45.71% a 49.10% sin reentrenamiento alguno. Para evitar ese sesgo, todas "
    "las métricas reportadas usan el conjunto de validación original."
))

# --- Curvas y matriz de confusión del entrenamiento ---
for archivo, titulo in [("results.png", "Curvas de entrenamiento"),
                        ("confusion_matrix_normalized.png", "Matriz de confusión"),
                        ("PR_curve.png", "Curva Precision–Recall")]:
    ruta = RUTA_V3 / archivo
    if ruta.exists():
        display(Markdown(f"### {titulo}"))
        display(Image(filename=str(ruta), width=750))

In [ ]:
from ultralytics import YOLO

m = YOLO(r"C:\Users\PAULA\runs\detect\runs_vid\yolo11_pseudococcidae_v3\weights\best.pt")

for sz in [640, 960, 1280]:
    r = m.val(data="dataset_final_v2/data.yaml", imgsz=sz, batch=4,
              workers=2, verbose=False, plots=False,
              project="test_res", name=f"res_{sz}", exist_ok=True)
    print(f"imgsz={sz}: mAP50={r.box.map50:.4f}  R={r.box.mr:.4f}")

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np

dims = [Image.open(p).size for p in list(Path("dataset_final_v2/train/images").glob("*.*"))[:200]]
print("Dimensiones más comunes:", {d: dims.count(d) for d in set(dims)})

lados = []
for f in list(Path("dataset_final_v2/train/labels").glob("*.txt"))[:400]:
    for l in f.read_text().splitlines():
        if l.strip():
            w, h = map(float, l.split()[3:5])
            lados.append(np.sqrt(w * h) * 640)
lados = np.array(lados)
print(f"Lado medio de caja: {lados.mean():.1f} px | mediana: {np.median(lados):.1f} px")
print(f"Cajas < 16px: {(lados < 16).mean():.1%}  | < 32px: {(lados < 32).mean():.1%}")

In [ ]:
from ultralytics import YOLO
m = YOLO(r"C:\Users\PAULA\runs\detect\runs_vid\yolo11_pseudococcidae_v3\weights\best.pt")

for iou in [0.5, 0.7, 0.85]:
    r = m.val(data="dataset_final_v2/data.yaml", imgsz=640, iou=iou, batch=4,
              workers=2, verbose=False, plots=False,
              project="test_nms", name=f"iou_{iou}", exist_ok=True)
    print(f"NMS iou={iou}: mAP50={r.box.map50:.4f}  R={r.box.mr:.4f}")

In [ ]:
from ultralytics import YOLO
from pathlib import Path

imgs = sorted(Path("dataset_final_v2/train/images").glob("*.*"))
con_cajas = [p for p in imgs if (Path("dataset_final_v2/train/labels") / (p.stem + ".txt")).stat().st_size > 0][:6]

YOLO(r"C:\Users\PAULA\runs\detect\runs_vid\yolo11_pseudococcidae_v3\weights\best.pt")  # solo para importar
import cv2, matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 3, figsize=(15, 10))
for a, p in zip(ax.ravel(), con_cajas):
    im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    h, w = im.shape[:2]
    for l in (Path("dataset_final_v2/train/labels") / (p.stem + ".txt")).read_text().splitlines():
        if l.strip():
            _, x, y, bw, bh = map(float, l.split()[:5])
            x1, y1 = int((x-bw/2)*w), int((y-bh/2)*h)
            cv2.rectangle(im, (x1, y1), (int((x+bw/2)*w), int((y+bh/2)*h)), (255,0,0), 2)
    a.imshow(im); a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import cv2, hashlib

SRC = Path(r"C:\Users\PAULA\Downloads\fotos_campo")
OUT = Path(r"C:\Users\PAULA\fotos_campo_preproc"); OUT.mkdir(exist_ok=True)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))   # <- tus parámetros

vistos, n = set(), 0
for p in sorted(SRC.glob("*.*")):
    if p.suffix.lower() not in [".jpg", ".jpeg", ".png"]: continue
    im = cv2.imread(str(p))
    if im is None: continue

    h = hashlib.md5(cv2.resize(im, (64, 64)).tobytes()).hexdigest()
    if h in vistos:
        print(f"[dup] {p.name}"); continue
    vistos.add(h)

    # recorte cuadrado central (la zona afectada suele estar al centro)
    hh, ww = im.shape[:2]
    lado = min(hh, ww)
    y0, x0 = (hh - lado)//2, (ww - lado)//2
    im = im[y0:y0+lado, x0:x0+lado]
    im = cv2.resize(im, (640, 640), interpolation=cv2.INTER_AREA)

    lab = cv2.cvtColor(im, cv2.COLOR_BGR2LAB)
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    im = cv2.bilateralFilter(cv2.cvtColor(lab, cv2.COLOR_LAB2BGR), 9, 75, 75)

    cv2.imwrite(str(OUT / f"{p.stem}.jpg"), im); n += 1

print(f"\n{n} imágenes únicas listas\n")

m = YOLO(r"C:\Users\PAULA\runs\detect\runs_vid\yolo11_pseudococcidae_v3\weights\best.pt")
res = m.predict(source=str(OUT), conf=0.15, imgsz=640, save=True,
                project=r"C:\Users\PAULA\evaluacion_campo", name="gran_chimu", exist_ok=True)

for r in res:
    confs = [f"{c:.2f}" for c in r.boxes.conf.tolist()]
    print(f"{Path(r.path).name}: {len(r.boxes)} det. {confs}")

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import cv2, numpy as np, torch
from torchvision.ops import nms

SRC = Path(r"C:\Users\PAULA\Downloads\fotos_campo")   # <- tus recortes manuales
OUT  = Path(r"C:\Users\PAULA\evaluacion_campo\final"); OUT.mkdir(parents=True, exist_ok=True)
m    = YOLO(r"C:\Users\PAULA\runs\detect\runs_vid\yolo11_pseudococcidae_v3\weights\best.pt")
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))   # <- tus parámetros

CONF = 0.15

def preproc(t):
    t = cv2.resize(t, (640, 640), interpolation=cv2.INTER_CUBIC)
    lab = cv2.cvtColor(t, cv2.COLOR_BGR2LAB)
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    return cv2.bilateralFilter(cv2.cvtColor(lab, cv2.COLOR_LAB2BGR), 9, 75, 75)

for p in sorted(SRC.glob("*.*")):
    if p.suffix.lower() not in [".jpg", ".jpeg", ".png"]: continue
    orig = cv2.imread(str(p))
    if orig is None: continue
    H, W = orig.shape[:2]
    cajas, scores = [], []

    # 1) imagen completa con TTA
    r = m.predict(preproc(orig), conf=CONF, imgsz=640, augment=True, verbose=False)[0]
    ex, ey = W / 640, H / 640
    for b, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()):
        cajas.append([b[0]*ex, b[1]*ey, b[2]*ex, b[3]*ey]); scores.append(float(c))

    # 2) cuadrantes con solape, cada uno ampliado a 640 con TTA
    ts = int(min(H, W) * 0.6)
    paso = int(ts * 0.5)
    for y in range(0, max(H - ts, 0) + 1, paso):
        for x in range(0, max(W - ts, 0) + 1, paso):
            t = orig[y:y+ts, x:x+ts]
            if t.size == 0: continue
            r = m.predict(preproc(t), conf=CONF, imgsz=640, augment=True, verbose=False)[0]
            e = ts / 640
            for b, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()):
                cajas.append([b[0]*e+x, b[1]*e+y, b[2]*e+x, b[3]*e+y]); scores.append(float(c))

    vis, n = orig.copy(), 0
    if cajas:
        keep = nms(torch.tensor(cajas, dtype=torch.float32),
                   torch.tensor(scores, dtype=torch.float32), 0.4)
        for i in keep.tolist():
            x1, y1, x2, y2 = map(int, cajas[i])
            cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 0, 255), 3)
            cv2.putText(vis, f"{scores[i]:.2f}", (x1, max(y1-8, 18)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            n += 1
    cv2.imwrite(str(OUT / p.name), vis)
    print(f"{p.name}: {n} detección(es)")

print(f"\nResultados en: {OUT}")

In [ ]:
from pathlib import Path
from IPython.display import Image, display

carpeta = Path(r"C:\Users\PAULA\evaluacion_campo\final")
archivos = sorted(carpeta.glob("*.*"))
print(f"{len(archivos)} archivos encontrados\n")

for p in archivos:
    print(p.name)
    display(Image(filename=str(p), width=650))

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import cv2, numpy as np, torch, hashlib
from torchvision.ops import nms

SRC = Path(r"C:\Users\PAULA\Downloads\fotos_campo")
OUT = Path(r"C:\Users\PAULA\evaluacion_campo\filtrado"); OUT.mkdir(parents=True, exist_ok=True)
m = YOLO(r"C:\Users\PAULA\runs\detect\runs_vid\yolo11_pseudococcidae_v3\weights\best.pt")
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

CONF     = 0.15
SAT_MAX  = 40    # saturación máxima para considerar "cera"  (sube si filtra de más)
VAL_MIN  = 170   # brillo mínimo                             (baja si filtra de más)
FRAC_MIN = 0.30  # % mínimo de píxeles cerosos en la caja

def preproc(t):
    t = cv2.resize(t, (640, 640), interpolation=cv2.INTER_CUBIC)
    lab = cv2.cvtColor(t, cv2.COLOR_BGR2LAB)
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    return cv2.bilateralFilter(cv2.cvtColor(lab, cv2.COLOR_LAB2BGR), 9, 75, 75)

def frac_cerosa(parche):
    if parche.size == 0: return 0.0
    hsv = cv2.cvtColor(parche, cv2.COLOR_BGR2HSV)
    mask = (hsv[:, :, 1] < SAT_MAX) & (hsv[:, :, 2] > VAL_MIN)
    return mask.mean()

vistos = set()
tot_antes = tot_despues = 0

for p in sorted(SRC.glob("*.*")):
    if p.suffix.lower() not in [".jpg", ".jpeg", ".png"]: continue
    orig = cv2.imread(str(p))
    if orig is None: continue
    h = hashlib.md5(cv2.resize(orig, (64, 64)).tobytes()).hexdigest()
    if h in vistos: continue
    vistos.add(h)

    H, W = orig.shape[:2]
    cajas, scores = [], []

    r = m.predict(preproc(orig), conf=CONF, imgsz=640, augment=True, verbose=False)[0]
    ex, ey = W/640, H/640
    for b, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()):
        cajas.append([b[0]*ex, b[1]*ey, b[2]*ex, b[3]*ey]); scores.append(float(c))

    ts, paso = int(min(H, W)*0.6), int(min(H, W)*0.3)
    for y in range(0, max(H-ts, 0)+1, paso):
        for x in range(0, max(W-ts, 0)+1, paso):
            t = orig[y:y+ts, x:x+ts]
            if t.size == 0: continue
            r = m.predict(preproc(t), conf=CONF, imgsz=640, augment=True, verbose=False)[0]
            e = ts/640
            for b, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()):
                cajas.append([b[0]*e+x, b[1]*e+y, b[2]*e+x, b[3]*e+y]); scores.append(float(c))

    vis, n_antes, n_desp = orig.copy(), 0, 0
    if cajas:
        keep = nms(torch.tensor(cajas, dtype=torch.float32),
                   torch.tensor(scores, dtype=torch.float32), 0.4)
        for i in keep.tolist():
            n_antes += 1
            x1, y1, x2, y2 = [max(0, int(v)) for v in cajas[i]]
            f = frac_cerosa(orig[y1:y2, x1:x2])
            if f < FRAC_MIN:
                continue                      # descartada por color
            cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 0, 255), 3)
            cv2.putText(vis, f"{scores[i]:.2f}", (x1, max(y1-8, 18)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            n_desp += 1

    tot_antes += n_antes; tot_despues += n_desp
    cv2.imwrite(str(OUT / f"{p.stem}.jpg"), vis)
    print(f"{p.name}: {n_antes} → {n_desp}")

print(f"\nTOTAL: {tot_antes} → {tot_despues} detecciones tras filtro cromático")
print(f"Resultados en: {OUT}")

In [ ]:
%matplotlib inline
from pathlib import Path
from IPython.display import Image, display

carpeta = Path(r"C:\Users\PAULA\evaluacion_campo\filtrado")
archivos = sorted([f for f in carpeta.glob("*") if f.suffix.lower() in (".jpg", ".jpeg", ".png")])

for f in archivos:
    print("=" * 60)
    print(f.name)
    display(Image(filename=str(f), width=700))

In [ ]:
from pathlib import Path

for base in ["dataset_final_v2", "dataset_final_v3"]:
    print(f"\n=== {base} ===")
    p = Path(base)
    if not p.exists():
        print("  no existe"); continue
    for sub in sorted(p.iterdir()):
        if sub.is_dir():
            n_img = len(list((sub / "images").glob("*.*"))) if (sub / "images").exists() else 0
            print(f"  {sub.name}: {n_img} imágenes")
    yaml = p / "data.yaml"
    if yaml.exists():
        print("  --- data.yaml ---")
        print("  " + yaml.read_text().replace("\n", "\n  "))

In [ ]:
from pathlib import Path

raices = [Path.home(), Path.home() / "Downloads"]
vistos = set()

for raiz in raices:
    for p in raiz.rglob("test"):
        if p.is_dir() and (p / "images").exists() and str(p) not in vistos:
            vistos.add(str(p))
            n = len(list((p / "images").glob("*.*")))
            print(f"{p}  →  {n} imágenes")

print("\n--- carpetas con data.yaml ---")
for raiz in raices:
    for y in raiz.rglob("data.yaml"):
        print(y.parent)